# Lab 06 — Graphological Feature Analysis

> **GraphoLab** | Forensic Graphology Laboratory

**Technique:** OpenCV + classical image processing  
**Task:** Automatically extract and visualise graphological features from handwriting  
**Forensic use case:** Profiling, comparative analysis, expert witness support

---

## What Are Graphological Features?

Forensic graphologists examine these measurable characteristics:

| Feature | Description | Forensic Relevance |
|---------|-------------|--------------------|
| **Letter slant** | Angle of strokes from vertical | Emotional state, consistency with known samples |
| **Stroke pressure** | Ink density / stroke thickness | Writer's physical characteristics, pen type |
| **Letter size** | Height and width of characters | Consistency across samples |
| **Word spacing** | Distance between words | Writing speed, layout habits |
| **Baseline** | Regularity of the text baseline | Emotional stability indicators |
| **Ink density** | Distribution of dark vs. light pixels | Pen pressure |


## GraphoLab Core — Quick Start

> The production implementation of graphological feature analysis is available in [`core/graphology.py`](../core/graphology.py).
> It measures slant, pressure, letter height/width, word spacing, and ink density using
> OpenCV + classical image processing, and returns both a Markdown report and an annotated image.
>
> Run the cell below to import it directly. The remaining cells implement the same analysis
> from scratch for educational purposes.

In [ ]:
# GraphoLab Core — production usage
# Run this cell to use the shared core module instead of the notebook implementation below.
import sys, pathlib
sys.path.insert(0, str(pathlib.Path("..").resolve()))

from core.graphology import grapho_analyse
from PIL import Image
import numpy as np

# Example: analyse graphological features of a handwriting sample
# doc = np.array(Image.open("../data/samples/handwritten_text_01.png").convert("RGB"))
# report, annotated_image = grapho_analyse(doc)
# print(report)
print("core.graphology imported — grapho_analyse() ready.")

## Setup


In [ ]:
# !pip install opencv-python numpy matplotlib scipy Pillow scikit-image

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional

import cv2
import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import ndimage

print("Libraries loaded.")

## Image Preprocessing Pipeline


In [ ]:
def load_handwriting(path: str | Path) -> np.ndarray:
    """Load a handwriting image as a grayscale numpy array."""
    img = Image.open(path).convert('L')
    return np.array(img)


def preprocess(gray: np.ndarray) -> np.ndarray:
    """Binarise handwriting: returns binary image with white strokes on black BG."""
    # Gaussian blur to reduce noise
    blurred = cv2.GaussianBlur(gray, (3, 3), 0)
    # OTSU binarisation — stroke pixels = 255, background = 0
    _, binary = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return binary


def deskew(binary: np.ndarray) -> np.ndarray:
    """Correct the global skew angle of the text."""
    coords = np.column_stack(np.where(binary > 0))
    if len(coords) < 10:
        return binary
    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45:
        angle = 90 + angle
    if abs(angle) < 0.5:
        return binary  # skip tiny corrections
    h, w = binary.shape
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    return cv2.warpAffine(binary, M, (w, h), flags=cv2.INTER_NEAREST,
                          borderMode=cv2.BORDER_CONSTANT, borderValue=0)


print("Preprocessing functions ready.")

## Feature Measurement Functions


In [ ]:
@dataclass
class GraphologicalFeatures:
    slant_angle_deg: float = 0.0          # Mean letter slant (degrees from vertical)
    slant_std_deg: float = 0.0            # Slant variation
    pressure_mean: float = 0.0            # Mean stroke darkness (0–255)
    pressure_std: float = 0.0
    letter_height_mean_px: float = 0.0
    letter_height_std_px: float = 0.0
    letter_width_mean_px: float = 0.0
    word_spacing_mean_px: float = 0.0
    ink_density: float = 0.0              # Fraction of ink pixels
    baseline_variance_px: float = 0.0    # Baseline regularity
    connected_components: int = 0


def measure_slant(binary: np.ndarray) -> tuple[float, float]:
    """Estimate letter slant from the orientation of stroke contours."""
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    angles = []
    for cnt in contours:
        if cv2.contourArea(cnt) < 20:
            continue
        if len(cnt) >= 5:
            _, _, angle = cv2.fitEllipse(cnt)
            # Convert to slant from vertical: 90° = vertical
            slant = angle - 90.0
            if -60 < slant < 60:  # filter outliers
                angles.append(slant)
    if not angles:
        return 0.0, 0.0
    return float(np.mean(angles)), float(np.std(angles))


def measure_pressure(gray: np.ndarray, binary: np.ndarray) -> tuple[float, float]:
    """Measure stroke 'pressure' as darkness of ink pixels."""
    ink_pixels = gray[binary > 0]
    if len(ink_pixels) == 0:
        return 0.0, 0.0
    # Invert: darker = higher pressure score
    pressure = 255 - ink_pixels
    return float(pressure.mean()), float(pressure.std())


def measure_connected_components(binary: np.ndarray) -> dict:
    """Analyse connected components (approximate letter strokes)."""
    num, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    # Filter by minimum area
    valid = stats[1:][stats[1:, cv2.CC_STAT_AREA] > 15]
    if len(valid) == 0:
        return {"count": 0, "height_mean": 0, "height_std": 0,
                "width_mean": 0, "baseline_y": []}

    heights = valid[:, cv2.CC_STAT_HEIGHT]
    widths = valid[:, cv2.CC_STAT_WIDTH]
    # Baseline: bottom edge of each component
    baseline_y = valid[:, cv2.CC_STAT_TOP] + heights

    return {
        "count": len(valid),
        "height_mean": float(heights.mean()),
        "height_std": float(heights.std()),
        "width_mean": float(widths.mean()),
        "baseline_y": baseline_y,
    }


def measure_word_spacing(binary: np.ndarray) -> float:
    """Estimate average word spacing from horizontal projection gaps."""
    # Project binary image onto horizontal axis
    h_proj = binary.sum(axis=0)  # sum along rows
    # Find gaps (zero columns = space between words)
    in_gap = False
    gap_widths = []
    gap_width = 0
    for v in h_proj:
        if v == 0:
            in_gap = True
            gap_width += 1
        else:
            if in_gap and gap_width > 5:  # minimum gap to count as word space
                gap_widths.append(gap_width)
            in_gap = False
            gap_width = 0
    return float(np.mean(gap_widths)) if gap_widths else 0.0


def extract_graphological_features(image_path: str | Path) -> GraphologicalFeatures:
    """Full graphological feature extraction pipeline."""
    gray = load_handwriting(image_path)
    binary = preprocess(gray)
    binary = deskew(binary)

    slant_mean, slant_std = measure_slant(binary)
    pressure_mean, pressure_std = measure_pressure(gray, binary)
    cc = measure_connected_components(binary)
    word_spacing = measure_word_spacing(binary)
    ink_density = (binary > 0).mean()
    baseline_var = float(np.var(cc["baseline_y"])) if len(cc["baseline_y"]) > 1 else 0.0

    return GraphologicalFeatures(
        slant_angle_deg=slant_mean,
        slant_std_deg=slant_std,
        pressure_mean=pressure_mean,
        pressure_std=pressure_std,
        letter_height_mean_px=cc["height_mean"],
        letter_height_std_px=cc["height_std"],
        letter_width_mean_px=cc["width_mean"],
        word_spacing_mean_px=word_spacing,
        ink_density=ink_density,
        baseline_variance_px=baseline_var,
        connected_components=cc["count"],
    )

print("Feature measurement functions ready.")

## Visualisation Dashboard


In [ ]:
def show_dashboard(image_path: str | Path, features: GraphologicalFeatures,
                   title: str = "Graphological Analysis") -> None:
    """Display a forensic graphology analysis dashboard."""
    gray = load_handwriting(image_path)
    binary = preprocess(gray)
    binary_deskewed = deskew(binary)

    fig = plt.figure(figsize=(16, 10))
    fig.suptitle(title, fontsize=14, fontweight='bold')
    gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

    # Original image
    ax0 = fig.add_subplot(gs[0, :])
    ax0.imshow(gray, cmap='gray')
    ax0.set_title("Original Handwriting Sample", fontsize=11)
    ax0.axis('off')

    # Binary
    ax1 = fig.add_subplot(gs[1, 0])
    ax1.imshow(binary_deskewed, cmap='gray')
    ax1.set_title("Binarised & Deskewed", fontsize=10)
    ax1.axis('off')

    # Horizontal projection
    ax2 = fig.add_subplot(gs[1, 1])
    h_proj = binary_deskewed.sum(axis=0)
    ax2.fill_between(range(len(h_proj)), h_proj, alpha=0.6, color='steelblue')
    ax2.set_title("Horizontal Projection\n(word spacing gaps)", fontsize=10)
    ax2.set_xlabel("Column", fontsize=8)
    ax2.set_ylabel("Ink pixels", fontsize=8)

    # Vertical projection
    ax3 = fig.add_subplot(gs[1, 2])
    v_proj = binary_deskewed.sum(axis=1)
    ax3.fill_betweenx(range(len(v_proj)), v_proj, alpha=0.6, color='coral')
    ax3.set_title("Vertical Projection\n(text lines)", fontsize=10)
    ax3.set_ylabel("Row", fontsize=8)
    ax3.set_xlabel("Ink pixels", fontsize=8)
    ax3.invert_yaxis()

    # Metrics bar chart
    ax4 = fig.add_subplot(gs[2, :])
    metric_names = [
        "Slant (°)",
        "Pressure (0–255)",
        "Letter height (px)",
        "Letter width (px)",
        "Word spacing (px)",
        "Ink density (%)",
        "Baseline var (px²)",
    ]
    metric_values = [
        features.slant_angle_deg,
        features.pressure_mean,
        features.letter_height_mean_px,
        features.letter_width_mean_px,
        features.word_spacing_mean_px,
        features.ink_density * 100,
        features.baseline_variance_px,
    ]
    colors = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0',
              '#FF9800', '#607D8B', '#795548']
    bars = ax4.barh(metric_names, metric_values, color=colors, alpha=0.8)
    ax4.set_title("Graphological Metrics", fontsize=11)
    for bar, val in zip(bars, metric_values):
        ax4.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
                 f"{val:.2f}", va='center', fontsize=9)
    ax4.set_xlabel("Value")

    plt.show()

print("Dashboard function ready.")

## Demo — Analyse a Handwriting Sample


In [ ]:
def make_synthetic_handwriting_for_analysis(seed: int = 42, size=(600, 150)) -> Path:
    """Generate a synthetic handwriting image for demo."""
    rng = np.random.RandomState(seed)
    arr = np.full((size[1], size[0]), 240, dtype=np.uint8)
    img = Image.fromarray(arr)
    draw = ImageDraw.Draw(img)

    slant_angle = (seed % 5 - 2) * 8  # varies by seed
    x = 20
    for _ in range(rng.randint(12, 20)):
        h = rng.randint(25, 45)
        dx = int(h * np.tan(np.radians(slant_angle)))
        base_y = 100 + rng.randint(-8, 8)
        draw.line([(x + dx, base_y - h), (x, base_y)], fill=30, width=2)
        draw.arc([(x, base_y - h // 2),
                  (x + 20, base_y)], start=0, end=180, fill=30, width=2)
        x += rng.randint(22, 35)
        if x > size[0] - 30:
            break

    # Add noise
    noise = rng.normal(0, 4, (size[1], size[0])).astype(np.int16)
    arr2 = np.clip(np.array(img, dtype=np.int16) + noise, 0, 255).astype(np.uint8)
    out = Path(f"../data/samples/handwritten_text_demo_{seed:02d}.png")
    out.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(arr2).save(out)
    return out


samples_dir = Path("../data/samples")
real_sample = samples_dir / "handwritten_text_01.png"

if real_sample.exists():
    sample_path = real_sample
    print(f"Using real sample: {sample_path}")
else:
    sample_path = make_synthetic_handwriting_for_analysis(seed=7)
    print(f"Using synthetic sample: {sample_path}")

features = extract_graphological_features(sample_path)

print("\n--- Graphological Features ---")
for f_name, f_val in features.__dict__.items():
    print(f"  {f_name:30s}: {f_val:.4f}" if isinstance(f_val, float) else
          f"  {f_name:30s}: {f_val}")

show_dashboard(sample_path, features, title=f"Graphological Analysis — {sample_path.name}")

## Demo — Compare Two Samples

Compare features from two different writers to highlight stylistic differences:


In [ ]:
sample_a_path = make_synthetic_handwriting_for_analysis(seed=1)
sample_b_path = make_synthetic_handwriting_for_analysis(seed=9)

feat_a = extract_graphological_features(sample_a_path)
feat_b = extract_graphological_features(sample_b_path)

# Comparison bar chart
metric_names = ["Slant (°)", "Pressure", "Letter height",
                "Letter width", "Word spacing", "Ink density %"]
vals_a = [feat_a.slant_angle_deg, feat_a.pressure_mean, feat_a.letter_height_mean_px,
          feat_a.letter_width_mean_px, feat_a.word_spacing_mean_px, feat_a.ink_density * 100]
vals_b = [feat_b.slant_angle_deg, feat_b.pressure_mean, feat_b.letter_height_mean_px,
          feat_b.letter_width_mean_px, feat_b.word_spacing_mean_px, feat_b.ink_density * 100]

x = np.arange(len(metric_names))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width / 2, vals_a, width, label='Sample A', color='steelblue', alpha=0.8)
ax.bar(x + width / 2, vals_b, width, label='Sample B', color='coral', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(metric_names, rotation=15, ha='right')
ax.set_title("Graphological Feature Comparison — Sample A vs. Sample B",
             fontsize=12, fontweight='bold')
ax.legend()
ax.set_ylabel("Value")
plt.tight_layout()
plt.show()

## Forensic Notes

- All measurements are in **pixel units** — normalise by scan resolution (DPI) when comparing samples scanned at different resolutions.
- **Slant measurement** is sensitive to image quality. High-resolution scans (≥300 DPI) are recommended.
- These features are most useful as **supporting evidence** alongside a qualified examiner's opinion, not as standalone proof.
- When comparing samples, always document: date of scan, resolution, scanner model, and any image processing applied.

---

**End of labs.** → See the [Interactive Gradio Demo](../app/grapholab_demo.py) to combine all four capabilities in a single browser interface.
